# Technical Data Cleaning – Banking Transaction Data

Notebook ini berisi proses pembersihan teknis data transaksi perbankan
sebelum dimuat ke PostgreSQL sebagai raw table.

In [1]:
import pandas as pd

DATA_PATH = "../data/raw/Bank_Transaction_Fraud_Detection.csv"
df = pd.read_csv(DATA_PATH)

## 1. Standarisasi Nama Kolom

In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns

Index(['customer_id', 'customer_name', 'gender', 'age', 'state', 'city',
       'bank_branch', 'account_type', 'transaction_id', 'transaction_date',
       'transaction_time', 'transaction_amount', 'merchant_id',
       'transaction_type', 'merchant_category', 'account_balance',
       'transaction_device', 'transaction_location', 'device_type', 'is_fraud',
       'transaction_currency', 'customer_contact', 'transaction_description',
       'customer_email'],
      dtype='object')

## 2. Konversi Tipe Data Tanggal dan Waktu

In [3]:
df["transaction_date"] = pd.to_datetime(
    df["transaction_date"],
    format="%d-%m-%Y",
    errors="coerce"
)

df["transaction_time"] = pd.to_datetime(
    df["transaction_time"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

## 3. Validasi Hasil Cleaning

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 24 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   customer_id              200000 non-null  object        
 1   customer_name            200000 non-null  object        
 2   gender                   200000 non-null  object        
 3   age                      200000 non-null  int64         
 4   state                    200000 non-null  object        
 5   city                     200000 non-null  object        
 6   bank_branch              200000 non-null  object        
 7   account_type             200000 non-null  object        
 8   transaction_id           200000 non-null  object        
 9   transaction_date         200000 non-null  datetime64[ns]
 10  transaction_time         200000 non-null  object        
 11  transaction_amount       200000 non-null  float64       
 12  merchant_id     

## 4. Kesimpulan Technical Cleaning

Berdasarkan proses technical data cleaning yang dilakukan:

- Seluruh nama kolom telah distandarisasi ke format **snake_case** untuk memastikan konsistensi skema
  dan kompatibilitas dengan PostgreSQL serta query SQL di tahap selanjutnya.
- Kolom `transaction_date` berhasil dikonversi ke tipe **datetime**, sehingga dapat digunakan
  untuk analisis berbasis waktu (harian, mingguan, bulanan) di database maupun BI tools.
- Tidak ditemukan missing values maupun duplikasi baris pada dataset ini, sehingga tidak diperlukan
  proses imputasi atau deduplikasi pada tahap technical cleaning.
- Seluruh data numerik (`transaction_amount`, `account_balance`, `age`, `is_fraud`) berada pada
  tipe data yang sesuai dan siap disimpan ke database tanpa kehilangan informasi.

Dataset hasil cleaning ini dianggap **siap secara teknis** untuk dimuat ke PostgreSQL sebagai
**raw table**, dan akan menjadi dasar untuk proses business transformation menggunakan SQL
pada tahap berikutnya.


## 5. Simpan Data Hasil Technical Cleaning

In [5]:
OUTPUT_PATH = "../data/processed/banking_transactions_clean.csv"

df.to_csv(OUTPUT_PATH, index=False)